# Resultados del entrenamiento

In [ ]:
import torch
import os
from data import TranslatorTokenizer, TranslationDataset
from transformer import Transformer, TransformerConfig
from config import VOCAB_PATH,RUNS_PATH

LONGITUD_CONTEXTO = 128
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vocab = os.path.join(VOCAB_PATH,"vocab_16k.json")
checkpoint_path = os.path.join(RUNS_PATH,"checkpoint_epoch_10_val_3.0146.pth")
DEVICE

device(type='cuda')

In [2]:
#Inicializamos Tokenizador y modelo
tokenizer = TranslatorTokenizer(path=vocab, context_length=LONGITUD_CONTEXTO)

config = TransformerConfig(
    vocab_size=len(tokenizer),
    pad_id=tokenizer.pad_id,
    context_length=LONGITUD_CONTEXTO
)

model = Transformer(config).to(DEVICE)
#Cargamos pesos del modelo
checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

In [3]:


# 5. Ponerlo en Modo Evaluación (APAGA EL DROPOUT, ESTO ES VITAL)
model.eval()
print("¡Modelo cargado y listo para traducir!")


# --- CREAMOS UNA FUNCIÓN CÓMODA PARA TRADUCIR ---

def traducir_frase(texto_ingles):
    # 1. Texto a números
    # Usamos pad=False porque para predecir 1 sola frase no necesitamos rellenar con ceros
    input_ids = tokenizer.encode(texto_ingles, pad=False)
    
    # 2. Convertir a Tensor y añadir la dimensión del Batch (B=1)
    # Pasa de [Ids] a [[Ids]]
    x_tensor = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(DEVICE)
    
    # 3. Usar tu propio método predict()
    y_pred_tensor = model.predict(
        x=x_tensor, 
        bos_id=tokenizer.start_id, 
        end_id=tokenizer.end_id, 
        max_new_tokens=LONGITUD_CONTEXTO, 
        device=DEVICE
    )
    
    # 4. El modelo devuelve un tensor con forma (1, longitud_generada). 
    # Seleccionamos la primera fila [0] para quitar el batch y decodificamos a texto.
    traduccion = tokenizer.decode(y_pred_tensor[0], skip_special_tokens=True)
    
    return traduccion

# ==========================================
# ¡A PROBARLO!
# ==========================================
frase_prueba = "Hello, how are you today? I am learning to program neural networks."
resultado = traducir_frase(frase_prueba)

print(f"\nEN: {frase_prueba}")
print(f"ES: {resultado}")

¡Modelo cargado y listo para traducir!

EN: Hello, how are you today? I am learning to program neural networks.
ES: Hola. Center, soy consciente de las redes sociales.
